# Kasus A — Deteksi Penipuan Pembukaan Rekening Bank (BAF Base, bulan 0–3)

**Tujuan:** memberi skor kemungkinan penipuan pada setiap pengajuan baru, lalu menangkap penipu sebanyak mungkin dengan syarat nasabah jujur yang ikut tertandai paling banyak 5%.

**Metrik utama:** TPR pada FPR 5% (persentase penipu tertangkap saat FPR dibatasi 5%). Metrik pendamping: ROC-AUC.

**Alur:** muat data → EDA singkat → bagi data berdasarkan waktu → baseline → tuning → latih final → evaluasi di validasi → simpan & muat ulang model.

> Letakkan `Base.csv` (dari Kaggle) di folder `data/`, lalu jalankan semua sel dari atas ke bawah.

In [ ]:
import json, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

from baf_utils import CAT_COLS, LABEL, NUM_COLS, clean_features, tpr_at_fpr

warnings.filterwarnings("ignore")
SEED = 42
FPR_TARGET = 0.05
DATA_PATH = Path("data/Base.csv")
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)

## 1. Muat data (hanya bulan 0–3)
File dibaca per potongan (chunk) lalu langsung disaring `month` 0–3 supaya hemat memori.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Base.csv tidak ditemukan. Unduh dari Kaggle (sgpjesus/bank-account-fraud-dataset-neurips-2022) "
        "dan simpan sebagai data/Base.csv"
    )

parts = []
for chunk in pd.read_csv(DATA_PATH, chunksize=200_000):
    parts.append(chunk[chunk["month"].between(0, 3)])
df = pd.concat(parts, ignore_index=True)
del parts

print("Bentuk data:", df.shape)
print("Bulan unik:", sorted(df["month"].unique()))
df.head(3)

## 2. EDA singkat

In [ ]:
summary = df.groupby("month")[LABEL].agg(pengajuan="size", penipuan="sum", rasio_penipuan="mean")
display(summary)
print(f"Rasio penipuan keseluruhan: {df[LABEL].mean():.4%}  (sangat tidak seimbang)")

# Nilai hilang pada BAF dikodekan -1 / negatif, bukan NaN
sentinel_cols = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
                 "session_length_in_minutes", "device_distinct_emails_8w"]
miss = {c: (df[c] == -1).mean() for c in sentinel_cols}
miss["intended_balcon_amount"] = (df["intended_balcon_amount"] < 0).mean()
print("\nProporsi nilai 'hilang' (kode -1 / negatif):")
display(pd.Series(miss, name="proporsi").sort_values(ascending=False).to_frame())

print("device_fraud_count unik:", df["device_fraud_count"].unique(), "-> konstan, dibuang")

## 3. Pembagian data latih / validasi

**Keputusan: pembagian berdasarkan waktu.** Data latih = `month` 0, 1, 2; data validasi = `month` 3.

**Alasan:**
1. Di produksi model memberi skor pengajuan *masa depan* dari model yang dilatih pada data *masa lalu*. Pembagian waktu meniru situasi ini; pembagian acak membuat pengajuan bulan yang sama bocor ke kedua sisi dan skor jadi terlalu optimis.
2. Pola penipuan berubah dari waktu ke waktu (drift), jadi kita ingin tahu seberapa baik model bertahan di bulan berikutnya.
3. Kolom `month` **tidak** dipakai sebagai fitur (tidak akan generalisasi ke bulan baru).

Untuk memilih hyperparameter tanpa menyentuh validasi: latih di bulan 0–1, uji di bulan 2 (tuning). Setelah konfigurasi terpilih, latih ulang di bulan 0–2 dan laporkan skor di bulan 3 sekali saja.

In [ ]:
train_df = df[df["month"] <= 2].reset_index(drop=True)
valid_df = df[df["month"] == 3].reset_index(drop=True)
tune_tr = train_df[train_df["month"] <= 1].reset_index(drop=True)
tune_va = train_df[train_df["month"] == 2].reset_index(drop=True)

for name, d in [("train (0-2)", train_df), ("valid (3)", valid_df), ("tune-train (0-1)", tune_tr), ("tune-valid (2)", tune_va)]:
    print(f"{name:18s} n={len(d):>7,}  penipu={d[LABEL].sum():>5,}  rasio={d[LABEL].mean():.3%}")

y_train, y_valid = train_df[LABEL].values, valid_df[LABEL].values
y_tune_tr, y_tune_va = tune_tr[LABEL].values, tune_va[LABEL].values

## 4. Model

- **Pra-pemrosesan** (`clean_features`): buang kolom non-fitur, ubah kode hilang (-1/negatif) menjadi `NaN`.
- **Model utama:** Gradient Boosting berbasis histogram (`HistGradientBoostingClassifier`). Alasannya: data tabular campuran, relasi non-linear dan interaksi antar-fitur, menangani `NaN` secara native (informasi "nilai hilang" ikut dipelajari), dan cepat untuk ratusan ribu baris.
- **Baseline:** Regresi Logistik (imputasi median + indikator hilang + standarisasi + one-hot) sebagai pembanding sederhana.
- Seluruh langkah dibungkus dalam satu `Pipeline`, sehingga model tersimpan bisa langsung menerima data mentah.

In [ ]:
def make_gbm(**params):
    pre = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), CAT_COLS),
         ("num", "passthrough", NUM_COLS)]
    )
    clf = HistGradientBoostingClassifier(
        categorical_features=list(range(len(CAT_COLS))),
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
        random_state=SEED, **params,
    )
    return Pipeline([("clean", FunctionTransformer(clean_features)), ("pre", pre), ("clf", clf)])


def make_logreg():
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1)
    return Pipeline([("clean", FunctionTransformer(clean_features)), ("pre", pre), ("clf", clf)])


def evaluate(model, X, y):
    s = model.predict_proba(X)[:, 1]
    tpr, thr = tpr_at_fpr(y, s, FPR_TARGET)
    return {"tpr@5%fpr": tpr, "auc": roc_auc_score(y, s), "threshold": thr}, s

## 5. Baseline dan tuning (latih bulan 0–1, uji bulan 2)

In [ ]:
configs = {
    "gbm_lr0.1_leaf31":           dict(learning_rate=0.1,  max_leaf_nodes=31, max_iter=500, l2_regularization=0.0),
    "gbm_lr0.05_leaf31":          dict(learning_rate=0.05, max_leaf_nodes=31, max_iter=800, l2_regularization=1.0),
    "gbm_lr0.05_leaf15":          dict(learning_rate=0.05, max_leaf_nodes=15, max_iter=800, l2_regularization=1.0),
    "gbm_lr0.05_leaf31_balanced": dict(learning_rate=0.05, max_leaf_nodes=31, max_iter=800, l2_regularization=1.0, class_weight="balanced"),
    "gbm_lr0.05_leaf63_minleaf100": dict(learning_rate=0.05, max_leaf_nodes=63, max_iter=800, l2_regularization=1.0, min_samples_leaf=100),
}

rows = []
base = make_logreg().fit(tune_tr, y_tune_tr)
m, _ = evaluate(base, tune_va, y_tune_va)
rows.append({"model": "logreg_baseline", **m})

for name, p in configs.items():
    mdl = make_gbm(**p).fit(tune_tr, y_tune_tr)
    m, _ = evaluate(mdl, tune_va, y_tune_va)
    rows.append({"model": name, **m, "n_iter": mdl.named_steps["clf"].n_iter_})
    print(f"{name:32s} TPR@5%FPR={m['tpr@5%fpr']:.4f}  AUC={m['auc']:.4f}")

tune_table = pd.DataFrame(rows).sort_values("tpr@5%fpr", ascending=False).reset_index(drop=True)
display(tune_table)
best_name = tune_table[tune_table["model"] != "logreg_baseline"].iloc[0]["model"]
print("Konfigurasi GBM terpilih:", best_name)

## 6. Latih final (bulan 0–2) dan evaluasi di validasi (bulan 3)

In [ ]:
final_model = make_gbm(**configs[best_name]).fit(train_df, y_train)
logreg_model = make_logreg().fit(train_df, y_train)

m_gbm, s_gbm = evaluate(final_model, valid_df, y_valid)
m_lr, s_lr = evaluate(logreg_model, valid_df, y_valid)

result = pd.DataFrame([{"model": "logreg_baseline", **m_lr}, {"model": f"GBM ({best_name})", **m_gbm}])
display(result)

# Selang kepercayaan bootstrap 95% untuk TPR@5%FPR model final
rng = np.random.default_rng(SEED)
boots = []
for _ in range(200):
    idx = rng.integers(0, len(y_valid), len(y_valid))
    if y_valid[idx].sum() == 0:
        continue
    boots.append(tpr_at_fpr(y_valid[idx], s_gbm[idx], FPR_TARGET)[0])
ci = np.percentile(boots, [2.5, 97.5])
print(f"\nTPR@5%FPR (validasi, bulan 3) = {m_gbm['tpr@5%fpr']:.4f}   95% CI bootstrap [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"ROC-AUC (validasi)            = {m_gbm['auc']:.4f}")

In [ ]:
threshold = m_gbm["threshold"]
flag = s_gbm >= threshold
honest, fraud = (y_valid == 0), (y_valid == 1)
print(f"Ambang skor        : {threshold:.5f}")
print(f"FPR (jujur tertandai): {flag[honest].mean():.4%}")
print(f"TPR (penipu tertangkap): {flag[fraud].mean():.4%}")
print(f"Presisi pada ambang : {fraud[flag].mean():.4%}  (dari {flag.sum():,} pengajuan yang ditandai)")

fpr, tpr, _ = roc_curve(y_valid, s_gbm)
fpr_l, tpr_l, _ = roc_curve(y_valid, s_lr)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(fpr, tpr, label=f"GBM (AUC={m_gbm['auc']:.3f})")
ax.plot(fpr_l, tpr_l, label=f"LogReg (AUC={m_lr['auc']:.3f})", alpha=.7)
ax.axvline(FPR_TARGET, color="gray", ls="--", lw=1)
ax.scatter([FPR_TARGET], [m_gbm["tpr@5%fpr"]], color="red", zorder=5, label=f"TPR@5%FPR={m_gbm['tpr@5%fpr']:.3f}")
ax.set_xlabel("FPR (nasabah jujur tertandai)"); ax.set_ylabel("TPR (penipu tertangkap)")
ax.set_title("ROC pada data validasi (bulan 3)"); ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig("roc_valid.png", dpi=150); plt.show()

## 7. Simpan model dan muat ulang

Model disimpan sebagai satu `Pipeline` (termasuk pembersihan fitur). `baf_utils.py` harus ada di folder yang sama saat memuat.
Ambang skor untuk FPR 5% disimpan terpisah di `models/baf_threshold.json`.

In [ ]:
MODEL_PATH = MODEL_DIR / "baf_gbm_pipeline.joblib"
joblib.dump(final_model, MODEL_PATH)

meta = {
    "model": f"HistGradientBoosting ({best_name})",
    "train_months": [0, 1, 2], "valid_month": 3,
    "fpr_target": FPR_TARGET,
    "threshold_at_fpr5": float(threshold),
    "valid_tpr_at_fpr5": float(m_gbm["tpr@5%fpr"]),
    "valid_auc": float(m_gbm["auc"]),
    "valid_tpr_ci95": [float(ci[0]), float(ci[1])],
    "baseline_logreg_valid_tpr_at_fpr5": float(m_lr["tpr@5%fpr"]),
}
(MODEL_DIR / "baf_threshold.json").write_text(json.dumps(meta, indent=2))

# Muat ulang dan pastikan skor identik
reloaded = joblib.load(MODEL_PATH)
s_reload = reloaded.predict_proba(valid_df)[:, 1]
assert np.allclose(s_reload, s_gbm), "Skor model hasil muat ulang berbeda!"
print("Model tersimpan di", MODEL_PATH, "dan skor setelah dimuat ulang identik.")
print(json.dumps(meta, indent=2))

## 8. Contoh pemakaian: memberi skor pengajuan baru

In [ ]:
def score_applications(raw_df: pd.DataFrame, model_path=MODEL_PATH, meta_path=MODEL_DIR / "baf_threshold.json"):
    """raw_df = DataFrame mentah berkolom seperti Base.csv (kolom fraud_bool/month boleh ada atau tidak)."""
    model = joblib.load(model_path)
    thr = json.loads(Path(meta_path).read_text())["threshold_at_fpr5"]
    out = pd.DataFrame({"fraud_score": model.predict_proba(raw_df)[:, 1]}, index=raw_df.index)
    out["flag_manual_review"] = out["fraud_score"] >= thr
    return out

demo = score_applications(valid_df.drop(columns=[LABEL]).head(10))
demo